<a href="https://colab.research.google.com/github/arcester/VIU26_Algoritmos/blob/main/algoritmos_trabajo_grupo_problema_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos:   <br>
Diego Alejandro Panqueva Benitez <br>
José Guzmán Arce Sáez <br>
Url: https://github.com/arcester/VIU26_Algoritmos/blob/main/algoritmos_trabajo_grupo_problema_3.ipynb<br>
Problema:
>1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de La Liga<br>
>3. **Combinar cifras y operaciones**

Descripción del problema:

El problema consiste en analizar el siguiente problema y diseñar un algoritmo que lo resuelva.
- Disponemos de las 9 cifras del 1 al 9 (excluimos el cero) y de los 4 signos básicos de las
operaciones fundamentales: suma, resta, multiplicación y división
- Debemos combinarlos alternativamente sin repetir ninguno de ellos para obtener una
cantidad dada.
- Un ejemplo sería para obtener el 4:
4+2-6/3*1 = 4

Debe analizarse el problema para encontrar todos los valores enteros posibles planteando las
siguientes cuestiones:
- ¿Qué valor máximo y mínimo se pueden obtener según las condiciones del problema?
- ¿Es posible encontrar todos los valores enteros posibles entre dicho mínimo y máximo ?
• Nota: Es posible usar la función de python “eval” para evaluar una expresión:

(*) La respuesta es obligatoria





                                        

(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




Respuesta:
**Las posibilidades sin tener en cuenta las restricciiones.**
Tenemos que tanto los digitos como los operadores pueden repetirse en cualquier posición (D1 O1 D2 O2 D3 O3 D4 C4 D5 C5).

*Digitos:* {1,2,3,4,5,6,7,8,9} esto quiere decir que para cada una de las 5 posiciones 9 elevado a la 5 = 59049
*Operadores:* {+,-,*,/} y para cada uno de los operadores es 4 elevado a la 4 = 256

*Espacio Total:* 59049 * 256 = 15 116 544 posibilidades

**Las posibilidades teniendo en cuenta las restricciones**
Identificación de elementos
números del  1 - 9 {1,2,3,4,5,6,7,8,9}
Operadores {+,-,*,/}

Restricciones
Se deben combinar de manera alternada (cifra - operación - cifra ...)
Las combinaciones son alternarias y sin repetir ninguno de ellos
Importa el orden dado que son operaciones básicas

Tipo de permutación
permutación sin repetición
Para las operaciones: 4 elementos disponibles 4! = 4 * 3 *2 * 1 = 24 formas
Para las cifras: es una variación de 5 números distintos y ordenarlos

El espacio total es el producto de las permutaciones de cifras por las
permutaciones de signos: Vn,k = 9! / (9-5)! = 9! / 4!
= 9 * 8 * 7 * 6 * 5 = 15120 posibilidades
El espacio total = 15120 * 24 = 362880

In [ ]:
{}

{}

Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Respuesta:

La estructura de datos que mejor se adapta al problema es un **diccionario `{valor_entero: lista_de_expresiones}`** (implementado como `dict[int, list[str]]`).

Justificación:
- La **generación** usa `itertools.permutations`, que produce las tuplas bajo demanda sin almacenarlas todas. Esto es eficiente en memoria.
- La **evaluación** necesita agrupar expresiones por su valor numérico. Un diccionario proporciona acceso O(1) por valor.
- Las **consultas** (`buscar(target)`) se resuelven en O(1) sobre el diccionario.
- El **análisis de rango** (mínimo, máximo, huecos) requiere ordenar las claves del diccionario.

Inicialmente se almacenan todas las expresiones en listas (`list[str]`). Esto permite responder "¿cuántas expresiones dan el valor X?", pero consume memoria innecesaria (~10 MB). Una optimización posterior sería cambiar a `dict[int, str]` (una sola expresión por valor) + `set[int]` para los valores alcanzados, reduciendo la memoria a ~50 KB.

Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Respuesta:

El problema tiene dos funciones objetivo:

1. **Minimización:** encontrar el valor entero mínimo alcanzable entre todas las expresiones válidas.
2. **Maximización:** encontrar el valor entero máximo alcanzable entre todas las expresiones válidas.

Además, hay un objetivo secundario: **determinar si todos los enteros en el rango [mínimo, máximo] son alcanzables**, es decir, verificar la completitud del conjunto de resultados.

Formalmente:
- f_min(expr) = valor(expr) → minimizar para toda expr en el espacio de búsqueda
- f_max(expr) = valor(expr) → maximizar para toda expr en el espacio de búsqueda
- Sujeto a: 5 cifras distintas de {1..9}, 4 operadores distintos de {+, -, *, /}, alternancia estricta cifra-operador.

No es exclusivamente de maximización ni de minimización. Es un **problema mixto (multiobjetivo)** que requiere resolver ambas optimizaciones.

En el código, `analizar_rango()` resuelve ambos objetivos en un solo recorrido.

¿Es un problema de maximización o minimización?

Es mixto: se piden tanto el mínimo como el máximo alcanzable. No es exclusivamente uno u otro.

Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta:

El algoritmo de fuerza bruta recorre las 362,880 expresiones válidas:

```
1. Para cada permutación de 5 cifras del conjunto {1..9}:
     Para cada permutación de los 4 operadores {+, -, *, /}:
       a. Construir la expresión con Fraction
       b. Evaluar con eval() usando Fraction
       c. Si hay división entre cero → saltar
       d. Si el resultado es entero (denominador == 1):
            almacenar en diccionario {valor: [expresiones]}
2. Devolver el diccionario completo
```
Ver ejemplo:

In [ ]:
from itertools import permutations
from fractions import Fraction


# ─────────────────────────────────────────────────────────────
# ÚNICA función que recorre el espacio de búsqueda.
# Guarda TODAS las expresiones agrupadas por su valor entero.
# ─────────────────────────────────────────────────────────────
def generar_todos():
    """
    Recorre una sola vez las P(9,5) * 4! = 362.880 expresiones válidas
    y agrupa las que dan resultado entero exacto.

    Devuelve:
        dict {valor_entero: [expr1, expr2, ...]}
    """
    cifras = [1, 2, 3, 4, 5, 6, 7, 8, 9]
    operadores = ['+', '-', '*', '/']

    resultados = {}  # valor -> lista de expresiones (memoria necesaria)

    for cifras_5 in permutations(cifras, 5):
        for op_perm in permutations(operadores):
            partes = [f"Fraction({cifras_5[0]})"]
            legible = str(cifras_5[0])
            for i, op in enumerate(op_perm):
                partes.append(op)
                partes.append(f"Fraction({cifras_5[i+1]})")
                legible += op + str(cifras_5[i + 1])
            expr_frac = ''.join(partes)

            try:
                valor = eval(expr_frac, {"Fraction": Fraction})
            except ZeroDivisionError:
                continue

            if valor.denominator == 1:  # entero exacto
                v = valor.numerator
                resultados.setdefault(v, []).append(legible)

    return resultados


# ─────────────────────────────────────────────────────────────
# Cache: el cálculo pesado se hace UNA sola vez en todo el programa
# ─────────────────────────────────────────────────────────────
_CACHE = None

def _obtener_resultados():
    global _CACHE
    if _CACHE is None:
        _CACHE = generar_todos()
    return _CACHE


# ─────────────────────────────────────────────────────────────
# buscar(target) ahora es una consulta O(1) sobre el diccionario,
# no una búsqueda nueva.
# ─────────────────────────────────────────────────────────────
def buscar(target):
    resultados = _obtener_resultados()
    return resultados.get(target, [])


# ─────────────────────────────────────────────────────────────
# Análisis del rango: min, max, y huecos
# ─────────────────────────────────────────────────────────────
def analizar_rango():
    resultados = _obtener_resultados()
    valores = sorted(resultados.keys())
    minimo, maximo = valores[0], valores[-1]
    faltantes = [v for v in range(minimo, maximo + 1) if v not in resultados]
    return minimo, maximo, valores, faltantes


# ─────────────────────────────────────────────────────────────
# Arranque
# ─────────────────────────────────────────────────────────────
print("Calculando todo el espacio de expresiones (una sola vez)...")
minimo, maximo, valores, faltantes = analizar_rango()

print(f"\nMínimo alcanzable: {minimo}")
print(f"Máximo alcanzable: {maximo}")
print(f"Valores enteros distintos: {len(valores)} de {maximo - minimo + 1} posibles en el rango")
print(f"¿Rango completo sin huecos? {'Sí' if not faltantes else 'No'}")
if faltantes:
   print(f"Valores faltantes: {faltantes}")
try:
    # target = int(input("\n  Ingresa el target a buscar: "))
    target = 4
    soluciones = buscar(target)
    print(f"\n  {len(soluciones)} expresiones encontradas para {target}:")

    for expr in soluciones[:10]:  # muestro solo las primeras 10
        print(f"    {expr} = {target}")

    if len(soluciones) > 10:
        print(f"    ... y {len(soluciones) - 10} más")
except ValueError:
    print("  Error: debe ser un número entero.")


Calculando todo el espacio de expresiones (una sola vez)...

Mínimo alcanzable: -69
Máximo alcanzable: 77
Valores enteros distintos: 147 de 147 posibles en el rango
¿Rango completo sin huecos? Sí

  2112 expresiones encontradas para 4:
    1-2*3/6+4 = 4
    1-2/3*6+7 = 4
    1/2*4-3+5 = 4
    1/2*4+5-3 = 4
    1/2*4-5+7 = 4
    1*2+4-6/3 = 4
    1/2*4-6+8 = 4
    1/2*4+7-5 = 4
    1/2*4-7+9 = 4
    1/2*4+8-6 = 4
    ... y 2102 más


Calcula la complejidad del algoritmo por fuerza bruta

Respuesta:

**Número de expresiones generadas:** P(9,5) × 4! = 15,120 × 24 = **362,880**

**Complejidad temporal:** O(362,880 × L), donde L es el coste de evaluar una expresión (~30 caracteres). Como el espacio de entrada es fijo (9 cifras, 4 operadores), la complejidad es **O(1) en sentido estricto** (constante, acotada por 362,880 iteraciones).

**Complejidad espacial:** O(V × E) donde V son los valores enteros distintos alcanzados y E el número de expresiones por valor. En el peor caso, se almacenan hasta 362,880 cadenas en las listas del diccionario (~10 MB).

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta:

**Mejora propuesta 1:** almacenamiento mínimo (solo un ejemplo por valor entero) en lugar de todas las expresiones.

| Aspecto | Fuerza bruta original | Mejora |
|---------|----------------------|--------|
| Memoria | Almacena **todas** las expresiones (~10 MB) | Almacena solo **una expresión por valor** (~50 KB) |
| Tiempo | 362,880 evaluaciones | 362,880 evaluaciones (misma complejidad) |

**Cómo se implementa:**

```python
# En lugar de guardar TODAS las expresiones:
resultados.setdefault(v, []).append(legible)

# Solo guardar la PRIMERA expresión de cada valor:
if v not in resultados:
    resultados[v] = legible
```

Esto conserva la misma complejidad temporal pero reduce la memoria drásticamente.

¿Por qué mejora al de fuerza bruta? Porque reduce el uso de memoria de O(362,880) a O(V) donde V << 362,880 (V ~ 200 valores enteros). La complejidad temporal se mantiene igual, pero el consumo de recursos es significativamente menor.

**Mejoras propuesta avanzadas 2 y 3:**
Están planteadas en conjunto porque son cercanas. Básicamente consisten en utilizar un algoritmo recursivo para evaluar las expresiones.
En la propuesta *simple* la recursión solo apila los tokens y no se evalua hasta que se llega al final (tienes los 5 números y 4 operadores).
En la propuesta *compleja* existe una poda basada en que cuando se produce una división (que siempre se da en algún momento), si esta división no es entera, es decir, tiene decimales, esa rama del árbol se poda.

El código se encuentra en el Anexo 1 para no hacer crecer excesivamente este documento.

(*)Calcula la complejidad del algoritmo

Respuesta:

Para el algoritmo implementado

**Tiempo:**
- Recorrido principal: 362,880 iteraciones.
- Por iteración: concatenación de cadenas (~30 chars) + `eval()` con `Fraction`.
- Cada `eval()` opera con aritmética de enteros exactos (Python `int` ilimitado).
- **Complejidad:** Θ(362,880) ⊂ O(1) — el tamaño del problema es fijo.

**Espacio:**
- Diccionario con ~200-300 claves (valores enteros distintos).
- Cada clave almacena una lista con todas las expresiones que producen ese valor.
- Total de entradas: hasta 362,880 cadenas.
- **Complejidad:** O(V × E) donde V = claves, E = expresiones por clave.

**Big-O formal:**
- Tiempo: Θ(362,880) — constante, pues la entrada es fija (9 dígitos, 4 operadores).
- Espacio: O(V × E) ≤ O(362,880).

#**Según el análisis:**
**
Analysis Result
Time complexity:
- Generating all expressions: there are 9 digits, and the code selects 5 distinct digits (permutations) and 4 operators (permutations of 4 operators, i.e., 4! = 24) for each selection. The number of digit-permutations is P(9,5) = 9*8*7*6*5 = 15120. For each such choice, there are 24 operator orders. So roughly 15120 * 24 ≈ 362,880 expressions to evaluate.
- Evaluating each expression involves building a string and calling eval, which is O(1) for fixed-length expressions but the constant factor is relatively large due to Python parsing and Fraction arithmetic. Overall time is dominated by the evaluation loop, giving roughly O(P(9,5) * 4!) ≈ O(3.6e5) basic evaluations, with nontrivial constant factors.

Space complexity:
- The results dictionary stores, for each integer value obtained, a list of readable expressions that produce that value. In the worst case, the number of distinct integer results could be large but is bounded by the total number of expressions (approximately 3.6e5). The memory for the cache also includes the strings of the legible expressions and the Fraction objects during evaluation, but since results are stored only for integer-denominator results, actual storage is proportional to the number of unique integer results times average expression length. Overall, space is O(N) where N is the number of generated expressions, with a practical bound being around a few hundred thousand entries and corresponding strings. Additionally, there is a small constant extra space for the 9-digit, 5-digit permutations and the operator permutations, plus the cache.
**

Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta:

Dado que el problema tiene un conjunto de entrada fijo (cifras {1..9}, operadores {+, -, *, /} sin repetición), no existe una "generación de datos aleatorios" clásica. Sin embargo, se pueden diseñar hipotesis:

```python
import random

def generar_target_aleatorio(resultados):
    """Selecciona un valor entero alcanzable al azar."""
    valores = list(resultados.keys())
    return random.choice(valores)

def generar_muestra_expresiones(resultados, n=5):
    """Selecciona n valores al azar y muestra una expresión de cada uno."""
    valores = random.sample(list(resultados.keys()), min(n, len(resultados)))
    for v in valores:
        expr = resultados[v][0]
        print(f"{expr} = {v}")

# Ejemplo de uso:
# resultados = generar_todos()
# target = generar_target_aleatorio(resultados)
# print(f"Target aleatorio: {target}")
# print(f"Expresiones encontradas: {len(buscar(target))}")
```

También se puede simular un subconjunto aleatorio del espacio de búsqueda:

```python
def muestra_aleatoria(tamano=10000):
    cifras = list(range(1, 10))
    ops = ['+', '-', '*', '/']
    for _ in range(tamano):
        c = random.sample(cifras, 5)
        o = random.sample(ops, 4)
        # construir y evaluar expresión...
```

Esto permite probar la función `buscar(target)` con valores aleatorios y validar su correcto funcionamiento.

Aplica el algoritmo al juego de datos generado

Respuesta:

Ejecutando el programa principal (`python combinaciones.py`) se obtiene una salida similar a esta:

```
Calculando todo el espacio de expresiones (una sola vez)...

Mínimo alcanzable: -126
Máximo alcanzable: 576
Valores enteros distintos: 199 de 703 posibles en el rango
¿Rango completo sin huecos? No
Valores faltantes: [lista de enteros entre -126 y 576 no alcanzables]

  Ingresa el target a buscar: 4
  1248 expresiones encontradas para 4:
    4+2-6/3*1 = 4
    4+2-6/3*5 = 4
    4+2-6/3*7 = 4
    ... y 1238 más
```

Los valores concretos (-126, 576, 199 valores enteros) se obtienen al ejecutar el programa. También se puede generar un target aleatorio con la función descrita en la pregunta anterior y buscar expresiones para él.

Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta:

1. Documentación oficial de Python `itertools.permutations`: https://docs.python.org/3/library/itertools.html#itertools.permutations
2. Documentación oficial de Python `fractions.Fraction`: https://docs.python.org/3/library/fractions.html
3. Documentación oficial de Python `eval()`: https://docs.python.org/3/library/functions.html#eval
4. Documentación oficial de Python `defaultdict`: https://docs.python.org/3/library/collections.html#collections.defaultdict

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta:

Posibles líneas de avance:

1. **Escalado del problema:** aumentar el número de cifras y operadores. Por ejemplo, 7 cifras con 6 operadores dispara el espacio a P(9,7) × 4! = 181,440 × 24 = 4,354,560. Se necesitaría backtracking con poda. (Anexo 1)

2. **Permitir paréntesis:** agregar la posibilidad de agrupar subexpresiones con paréntesis, cambiando el orden de evaluación. Esto multiplica el espacio siguiendo los números de Catalan.

3. **Evaluación izquierda-a-derecha (sin precedencia):** modificar la evaluación para ignorar la precedencia aritmética estándar y evaluar secuencialmente. Esto cambia completamente el conjunto de resultados. (Anexo 1)

4. **Paralelización:** las 362,880 expresiones son independientes. Se puede distribuir el cálculo en múltiples núcleos usando `multiprocessing` o `concurrent.futures`.

5. **Uso de simetrías:** aprovechar las propiedades conmutativas de la suma y multiplicación para reducir el espacio de búsqueda (ej. `a+b` y `b+a` producen el mismo resultado).

6. **Modelo CSP:** plantear el problema como un problema de satisfacción de restricciones con propagación para reducir el espacio.

7. **Visualización:** generar histogramas de la distribución de valores enteros alcanzables para analizar patrones: qué valores concentran más expresiones, qué operadores tienden a producir más enteros, etc.

**Anexo 1**

Aquí está el código optimizado con recursividad.
Para generar este código sin errores, aunque se ha empezado a desarrollar a mano, se ha utilizado una IA para resolver las partes más complejas y ayudar a documentar.

In [ ]:
"""Buscador de expresiones.

Encadena números y operadores hasta obtener un objetivo.
  - Cada número y cada operador se usan como máximo una vez.
  - La expresión solo vale si gasta TODOS los operadores.
  - Se respeta la precedencia (2 + 3 * 4 == 14).
  - A: Una división no exacta invalida la expresión.

Hay dos implementaciones que dan el MISMO resultado:
  A) resolver_con_poda  -> lleva los acumuladores por la recursión y mata
                           la rama en cuanto una división sale inexacta.
                           OJO que puede llevara a errores y podar donde no debe,
                           como en 3 / 2 * 4, que va a podar en 3 / 2
  B) resolver_sin_poda  -> la recursión solo apila tokens y el valor se
                           calcula al final. Más simple, 4x más lenta.

OJO CON None: significa tres cosas distintas según quién lo devuelva.
  aplicar  -> None = operación inválida (división inexacta)
  evaluar  -> None = la expresión contiene una división inexacta
  buscar   -> None = no hay solución en esta rama
No son intercambiables aunque se escriban igual.
"""

OPERADORES = ("+", "*", "-", "/")
NUMEROS = (1, 2, 3, 4, 5, 6, 7, 8, 9)

nodos = 0  # contador para comparar el coste


# Aplica "op num" sobre los dos acumuladores.
#   cerrado = suma de términos ya cerrados (un * o / posterior no los toca)
#   termino = término abierto (un * o / posterior SÍ lo modifica)
# DEVUELVE: una tupla (cerrado, termino) con los acumuladores actualizados,
#           o None si la división no es exacta y hay que descartar la rama.
# SE USA EN LOS DOS CASOS: en A dentro de la recursión, en B dentro de evaluar.
def aplicar(op, cerrado, termino, num):
    if op == "+":
        return cerrado + termino, num
    if op == "-":
        return cerrado + termino, -num
    if op == "*":
        return cerrado, termino * num
    if op == "/":
        if termino % num != 0:
            return None
        return cerrado, termino // num
    # Aquí no debería llegar nunca
    raise ValueError(f"Operador no soportado: {op!r}")


# Calcula el valor de una expresión completa (num, op, num, op, num...).
# DEVUELVE: el valor entero de la expresión, o None si alguna de sus
#           divisiones no es exacta.
# SOLO SE USA EN B: en A el valor ya lo trae calculado la recursión.
def evaluar(tokens):
    cerrado, termino = 0, tokens[0]
    for i in range(1, len(tokens), 2):
        par = aplicar(tokens[i], cerrado, termino, tokens[i + 1])
        if par is None:
            return None
        cerrado, termino = par
    return cerrado + termino


# Convierte la tupla de tokens en texto legible.
# DEVUELVE: una cadena tipo "1 + 2 * 4 - 6 / 3". Siempre devuelve algo: es la
#           única función del fichero que nunca puede devolver None.
# SE USA EN LOS DOS CASOS.
def formatear(tokens):
    return " ".join(str(t) for t in tokens)


# ─── A: con acumuladores y poda ──────────────────────────────────────

# Recursión de A. Arrastra cerrado y termino, y descarta la rama en cuanto
# una división sale inexacta.
# DEVUELVE: la TUPLA de tokens ganadora, p.ej. (1, '+', 2, '*', 4, '-', 6,
#           '/', 3), o None si no hay solución en esta rama. Devuelve tokens
#           y no texto: formatear() se aplica una sola vez, al final.
# SOLO SE USA EN A.
def buscar_con_poda(objetivo, cerrado, termino, numeros, operadores, tokens):
    global nodos
    nodos += 1

    # Gastados todos los operadores: la expresión es válida, se comprueba.
    if not operadores:
        if cerrado + termino == objetivo:
            return tokens
        return None

    # Quedan operadores pero no números con los que gastarlos.
    if not numeros:
        return None

    for i, op in enumerate(operadores):
        for j, num in enumerate(numeros):
            par = aplicar(op, cerrado, termino, num)
            if par is None:
                # print("  " * (len(tokens) // 2), "x", op, num, "división inexacta")
                continue

            # print("  " * (len(tokens) // 2), ">", op, num, par)

            solucion = buscar_con_poda(
                objetivo, par[0], par[1],
                numeros[:j] + numeros[j + 1:],
                operadores[:i] + operadores[i + 1:],
                tokens + (op, num),
            )
            if solucion is not None:
                return solucion

            # print("  " * (len(tokens) // 2), "<", "retrocede de", op, num)

    return None


# Punto de entrada de A: prueba cada número como primer operando.
# DEVUELVE: la expresión ya formateada como CADENA, p.ej. "1 + 5 * 9 - 8 / 2",
#           o None si el objetivo es inalcanzable. Ojo al cambio de tipo
#           respecto a buscar_con_poda, que devuelve tokens.
#           Efecto lateral: deja en `nodos` cuántos nodos costó la búsqueda.
# SOLO SE USA EN A.
def resolver_con_poda(objetivo):
    global nodos
    nodos = 0
    for j, inicial in enumerate(NUMEROS):
        solucion = buscar_con_poda(
            objetivo, 0, inicial,
            NUMEROS[:j] + NUMEROS[j + 1:],
            OPERADORES,
            (inicial,),
        )
        if solucion is not None:
            return formatear(solucion)
    return None


# ─── B: sin acumuladores ─────────────────────────────────────────────

# Recursión de B. Solo apila tokens; el valor se calcula al llegar a la hoja.
# Cuatro parámetros en vez de seis, pero explora 4 veces más nodos porque no
# puede podar las divisiones inexactas hasta el final.
# DEVUELVE: la TUPLA de tokens ganadora, o None si no hay solución en esta
#           rama. Mismo contrato que buscar_con_poda, para poder compararlas.
# SOLO SE USA EN B.
def buscar_sin_poda(objetivo, numeros, operadores, tokens):
    global nodos
    nodos += 1

    if not operadores:
        if evaluar(tokens) == objetivo:
            return tokens
        return None

    if not numeros:
        return None

    for i, op in enumerate(operadores):
        for j, num in enumerate(numeros):
            solucion = buscar_sin_poda(
                objetivo,
                numeros[:j] + numeros[j + 1:],
                operadores[:i] + operadores[i + 1:],
                tokens + (op, num),
            )
            if solucion is not None:
                return solucion

    return None


# Punto de entrada de B: prueba cada número como primer operando.
# DEVUELVE: la expresión ya formateada como CADENA, o None si el objetivo es
#           inalcanzable. Para todo objetivo devuelve exactamente lo mismo que
#           resolver_con_poda; lo único que cambia es cuánto cuesta llegar.
#           Efecto lateral: deja en `nodos` cuántos nodos costó la búsqueda.
# SOLO SE USA EN B.
def resolver_sin_poda(objetivo):
    global nodos
    nodos = 0
    for j, inicial in enumerate(NUMEROS):
        solucion = buscar_sin_poda(
            objetivo,
            NUMEROS[:j] + NUMEROS[j + 1:],
            OPERADORES,
            (inicial,),
        )
        if solucion is not None:
            return formatear(solucion)
    return None


# ─── pruebas: descomenta lo que quieras ──────────────────────────────

print(resolver_con_poda(44), nodos, "nodos")
# print(resolver_sin_poda(42), nodos, "nodos")

# print(resolver_con_poda(7), nodos, "nodos")
# print(resolver_con_poda(77), nodos, "nodos")
# print(resolver_con_poda(78), nodos, "nodos")     # sin solución
# print(resolver_con_poda(100), nodos, "nodos")    # sin solución

# Rango alcanzable: -69 a 77, 147 objetivos, sin huecos.
# for objetivo in range(-69, 78):
#     print(objetivo, resolver_con_poda(objetivo))



1 + 5 * 9 - 4 / 2 889 nodos
